# The 1.01 — Bijan, Gibbs or Allen?

Sleeper, 14-team superflex, half-PPR, seat 1. One question: **who is the optimal first pick?**

The answer has two halves, and they are not equally firm.

- **Not the quarterback.** Josh Allen is the best player of the three by a distance and still the
  wrong pick, because a superflex league lets you buy a starting quarterback at pick 28 and does
  not let you buy a running back of this class anywhere. That result is robust — it survives every
  opening plan tested below, and it is the one thing here the draft simulation genuinely knows.
- **Between the two backs, the board says Gibbs and the evidence says it's a coin flip.** The
  simulation prefers Gibbs in 93% of rooms, but that number turns out to be a mechanical restatement
  of the consensus projection rather than independent support for it — and the projection's whole
  Gibbs-over-Bijan margin rests on a 2025 touchdown rate that this warehouse can show does not
  persist. Every signal that isn't the projection points at Bijan.

Everything below is computed from the warehouse, not typed in. Re-run after a rebuild and the
findings update; if a number here moved, the draft moved.

In [1]:
import re
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from src.query import q
from src.gold.draft_plan import simulate_first_pick

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)

LEAGUE = "sleeper"
MY_SEAT = 1
CANDIDATES = {"Jahmyr Gibbs": "RB", "Bijan Robinson": "RB", "Josh Allen": "QB"}


def plan_slots(plan):
    """['QB', 'QB', 'RB', 'RB', 'TE'] for a composition label like '2QB2RB1TE'.

    The ADP control's label doesn't parse, and yields [] — which is what filters it out of
    every candidate comparison below, since a control has no composition to force.
    """
    return [position
            for count, position in re.findall(r"(\d)([A-Z]+)", plan)
            for _ in range(int(count))]

# Derived, never hard-coded: the league changed format once already this preseason.
settings = q("SELECT * FROM league_settings WHERE league_key = ?", [LEAGUE]).iloc[0]
TEAMS = int(settings.team_count)
print(f"{LEAGUE}: {TEAMS} teams, {settings.rec_pts} PPR, "
      f"{settings.superflex_slots} superflex slot(s), {settings.bench_slots} bench")

sleeper: 14 teams, 0.5 PPR, 1 superflex slot(s), 5 bench


## Is the first pick even the decision?

Worth settling before spending a notebook on it, because the roster-construction work
(`draft_strategy.ipynb`) found that in this superflex league *composition* — how many quarterbacks,
backs and receivers you open with — is the largest effect in the warehouse. If the pick is noise
next to the plan, the honest answer is "take anyone and get the plan right".

Below: the spread across opening plans from this seat, against the spread across the three
candidates (computed further down, quoted here for scale).

In [2]:
plans = q('''
    SELECT plan, round(points_vs_field, 1) AS vs_field,
           round(points_vs_field_stderr, 2) AS stderr
    FROM draft_plans
    WHERE league_key = ? AND draft_slot = ?
    ORDER BY points_vs_field DESC
''', [LEAGUE, MY_SEAT])

# "Plausible" = the openings a drafter would actually weigh, not 5WR. Everything that fields a
# quarterback and at least one back, which is the shape every candidate here can reach.
def reachable(plan):
    slots = plan_slots(plan)
    return "QB" in slots and "RB" in slots

plausible = plans[plans.plan.apply(reachable)]

print(f"all {len(plans)} openings          : {plans.vs_field.max() - plans.vs_field.min():6.1f} pts "
      f"({plans.iloc[0].plan} -> {plans.iloc[-1].plan})")
print(f"the {len(plausible)} plausible openings  : "
      f"{plausible.vs_field.max() - plausible.vs_field.min():6.1f} pts")
print(f"top 8 openings only        : "
      f"{plans.head(8).vs_field.max() - plans.head(8).vs_field.min():6.1f} pts")
pd.concat([plans.head(5), plans.tail(3)])

all 37 openings          :  305.4 pts (2QB2RB1TE -> 5WR)
the 15 plausible openings  :   87.5 pts
top 8 openings only        :   35.8 pts


,plan,vs_field,stderr
0,2QB2RB1TE,118.8,2.99
1,2QB2RB1WR,113.6,2.88
2,2QB3RB,105.5,3.36
3,2QB1RB1WR1TE,101.2,3.11
4,1QB2RB1WR1TE,91.8,4.05
34,4WR1TE,-159.2,6.69
35,3WR2TE,-161.9,6.87
36,5WR,-186.6,6.15


**Both are decisions, and they are not independent.** Composition is worth far more than the pick —
hundreds of points across the full menu, dozens across openings anyone would seriously consider,
against a first-pick spread of about fourteen. So the earlier research holds: get the plan right
first.

But the pick is not free of the plan. Taking Allen at 1.01 *commits* a quarterback slot in an
opening that only has five picks in it, and the section on the plan grid below tests whether that
commitment changes which plan you can afford. Fourteen points is also not nothing when the top
plans are separated by five.

## What the board says about the three in isolation

`draft_board` prices each player against a freely-available replacement at his own position, which
is the right unit for comparing across positions — 380 quarterback points and 315 running back
points are not the same currency until you subtract what the position gives away for free.

The individual sources are shown alongside the blend rather than behind it, because the blend hides
the one disagreement that matters here.

In [3]:
board = q('''
    SELECT player_name, position, team, bye_week,
           round(ol_grade, 1) AS ol_grade, ol_tier,
           espn_points, sleeper_points, cbs_points, fftoday_points, inhouse_points, num_sources,
           round(projected_points_adjusted, 1) AS projected,
           position_rank, round(replacement_level_points, 1) AS replacement,
           round(points_over_replacement, 1) AS por,
           consensus_adp, adp_stdev, adp_high, adp_low, round(availability, 3) AS availability
    FROM draft_board
    WHERE league_key = ? AND player_name IN ('Jahmyr Gibbs', 'Bijan Robinson', 'Josh Allen')
    ORDER BY points_over_replacement DESC
''', [LEAGUE])
board.set_index("player_name").T

player_name,Jahmyr Gibbs,Bijan Robinson,Josh Allen
position,RB,RB,QB
team,DET,ATL,BUF
bye_week,6,11,7
ol_grade,66.1,83.9,91.9
ol_tier,Q3,Q4 best,None
espn_points,330.637835,314.79373,369.690124
sleeper_points,347.03,335.15,392.18
cbs_points,342.25,312.05,410.55
fftoday_points,331.4,324.9,384.2
inhouse_points,284.71988,286.162366,356.127376


Points over replacement ranks them **Gibbs, Bijan, Allen** — the quarterback is third despite
projecting sixty-odd points more than either back, because superflex pushes 28 quarterbacks into
starting lineups and the 28th is a much better free player than the 31st running back.

Two things to carry forward. The four external sources all have Gibbs ahead of Bijan; the in-house
model has them level, and marginally the other way. And a board cannot settle a draft pick anyway —
it prices players one at a time, and the question is what each choice does to a whole roster.

## Why the quarterback is third: the pick you would make instead

The cost of a first-round pick is not the player's value, it is the value of the player you would
otherwise have taken at the same slot. From seat 1 the next picks are 28 and 29, so the real
comparison is not Allen against Gibbs — it is **Allen plus the best back left at 29** against
**Gibbs plus the best quarterback left at 29**.

`draft_availability` gives each player's probability of surviving to a pick. Treating those as
independent (they are marginal, one player at a time — the caveat is in `draft_plan`'s docstring)
turns them into an expected best-available at each position.

In [4]:
def expected_best_available(pick, position, league=LEAGUE, slot=MY_SEAT):
    """E[points over replacement of the best survivor at `position`], under independence.

    Walk the position's players from best to worst: each is the best available exactly when he
    survives and everyone above him did not.
    """
    pool = q('''
        SELECT points_over_replacement AS por, p_available
        FROM draft_availability
        WHERE league_key = ? AND draft_slot = ? AND overall_pick = ? AND position = ?
        ORDER BY points_over_replacement DESC
    ''', [league, slot, pick, position])
    still_gone = 1.0
    expected = 0.0
    for por, p in zip(pool.por, pool.p_available):
        expected += por * p * still_gone
        still_gone *= 1 - p
    return expected

replacement = q('''
    SELECT position, max(starters_at_position) AS starters,
           round(max(replacement_level_points), 1) AS replacement_level
    FROM draft_board WHERE league_key = ? GROUP BY position ORDER BY replacement_level DESC
''', [LEAGUE])
replacement["best_at_28"] = [round(expected_best_available(28, p), 1) for p in replacement.position]
replacement["best_at_29"] = [round(expected_best_available(29, p), 1) for p in replacement.position]
replacement

,position,starters,replacement_level,best_at_28,best_at_29
0,QB,28,232.8,74.1,73.6
1,WR,39,138.3,77.6,74.5
2,RB,31,133.1,94.6,93.6
3,K,14,130.0,22.0,22.0
4,TE,14,117.0,76.9,76.9
5,DST,14,100.3,40.2,40.2


In [5]:
# The two paths differ by exactly one swap: both plans spend picks 1 and 29 on a QB and a RB, in
# opposite orders, and every other pick in the opening is identical. So the comparison is clean.
por = board.set_index("player_name").por
swap = pd.DataFrame([
    {"first pick": name,
     "his por": por[name],
     "then at 29": "best RB left" if pos == "QB" else "best QB left",
     "worth": round(expected_best_available(29, "RB" if pos == "QB" else "QB"), 1)}
    for name, pos in CANDIDATES.items()
])
swap["pair total"] = (swap["his por"] + swap["worth"]).round(1)
swap.sort_values("pair total", ascending=False)

,first pick,his por,then at 29,worth,pair total
0,Jahmyr Gibbs,181.7,best QB left,73.6,255.3
1,Bijan Robinson,172.2,best QB left,73.6,245.8
2,Josh Allen,145.7,best RB left,93.6,239.3


**The quarterback loses the swap.** Allen is worth more than either back on his own, but the back
you can still get at 29 is worth appreciably more than the quarterback you can still get at 29 —
superflex demand runs 28 deep, so the position stays stocked exactly as long as you need it to.
Trading Allen for a back costs you the difference between two elite backs and gains you the
difference between Allen and a startable quarterback, and the second gap is smaller.

This is arithmetic on static prices, so it over-states the margin — it cannot see lineup slots,
bench, or bye weeks. The simulation below can, and lands in the same direction with a smaller
number.

## The paired simulation

`simulate_first_pick` forces each candidate at 1.01 and plays out the draft: every team drafts from
a board whose order is resampled from each player's own ADP distribution, rosters are filled to the
league's real lineup, and the season is scored on a hindsight-optimal lineup. Every candidate is run
through the **identical** sampled rooms, so the comparison is paired — a candidate's advantage is
measured room by room rather than by averaging two independent samples.

Each candidate gets the opening that suits him: the best plan from the table above, ordered so his
own position comes first.

In [6]:
PRIORITY = ["QB", "RB", "TE", "WR"]

def sequence(plan, first_position):
    """The five-round order for a composition, with `first_position` taken at 1.01.

    The rest follow in scarcity order. Ordering is a tiebreaker rather than a lever
    (see draft_strategy.ipynb), but it has to be *consistent* across candidates or the
    comparison stops being paired.
    """
    remaining = plan_slots(plan)
    remaining.remove(first_position)
    return [first_position] + sorted(remaining, key=PRIORITY.index)

BEST_PLAN = plans.iloc[0].plan
TRIALS = 3000
openings = {name: sequence(BEST_PLAN, pos) for name, pos in CANDIDATES.items()}
print(f"best opening: {BEST_PLAN}")
for name, seq in openings.items():
    print(f"  {name:<16} {' '.join(seq)}")

first_pick = simulate_first_pick(LEAGUE, MY_SEAT, openings, trials=TRIALS)
wide = first_pick.pivot(index="trial", columns="player_name", values="points_vs_field")

summary = pd.DataFrame({
    "points_vs_field": wide.mean().round(1),
    "stderr": (wide.std() / np.sqrt(TRIALS)).round(2),
    "room_to_room_sd": wide.std().round(1),
    "win_rate": first_pick.groupby("player_name").finish_rank.apply(lambda r: (r == 1).mean()).round(3),
    "avg_finish": first_pick.groupby("player_name").finish_rank.mean().round(2),
}).sort_values("points_vs_field", ascending=False)
summary

best opening: 2QB2RB1TE
  Jahmyr Gibbs     RB QB QB RB TE
  Bijan Robinson   RB QB QB RB TE
  Josh Allen       QB QB RB RB TE


,points_vs_field,stderr,room_to_room_sd,win_rate,avg_finish
player_name,,,,,
Jahmyr Gibbs,127.2,0.91,50.1,0.488,2.15
Bijan Robinson,117.1,0.92,50.2,0.409,2.41
Josh Allen,113.5,0.94,51.5,0.376,2.58


In [7]:
pairs = []
for a, b in [("Jahmyr Gibbs", "Bijan Robinson"), ("Jahmyr Gibbs", "Josh Allen"),
             ("Bijan Robinson", "Josh Allen")]:
    difference = wide[a] - wide[b]
    t, p = stats.ttest_rel(wide[a], wide[b])
    pairs.append({
        "comparison": f"{a} - {b}",
        "mean": round(difference.mean(), 2),
        "stderr": round(difference.std() / np.sqrt(TRIALS), 2),
        "t": round(t, 1),
        "p": f"{p:.1e}",
        "rooms won": round((difference > 0).mean(), 3),
        "10th pct": round(difference.quantile(0.10), 1),
        "90th pct": round(difference.quantile(0.90), 1),
    })
pd.DataFrame(pairs)

,comparison,mean,stderr,t,p,rooms won,10th pct,90th pct
0,Jahmyr Gibbs - Bijan Robinson,10.15,0.14,71.4,0.0e+00,0.930,2.6,17.4
1,Jahmyr Gibbs - Josh Allen,13.70,0.42,32.5,3.0e-199,0.790,-15.2,38.2
2,Bijan Robinson - Josh Allen,3.56,0.41,8.6,9.5e-18,0.624,-25.2,27.7


**Both backs beat Allen, and Gibbs beats Bijan.** All three gaps clear significance by margins that
would normally end an argument.

They should not end this one, for a reason the percentile columns hint at. Read the *spread*, not
just the mean: the Gibbs-over-Allen edge is worth about fourteen points on average but the tenth
percentile is negative — Allen genuinely wins a fifth of rooms. And the Gibbs-over-Bijan row has a
standard error a third the size of the others, which is the tell that the next section chases down.

## Does the answer survive the plan you run afterwards?

The comparison above gave every candidate the same, best, composition. That is the fair test of the
pick, but it leaves the interaction unmeasured: taking Allen spends one of five opening picks on a
quarterback, and maybe that is only affordable inside the plans that wanted two quarterbacks anyway.

So: every plausible opening, each candidate forced at 1.01, all through identical rooms.

In [8]:
GRID_TRIALS = 1500
GRID_PLANS = [p for p in plans.plan.head(12) if reachable(p)]

grid_rows = []
for plan in GRID_PLANS:
    result = simulate_first_pick(
        LEAGUE, MY_SEAT,
        {name: sequence(plan, pos) for name, pos in CANDIDATES.items()},
        trials=GRID_TRIALS,
    )
    result["composition"] = plan
    grid_rows.append(result)
grid = pd.concat(grid_rows, ignore_index=True)

by_plan = grid.pivot_table(index="composition", columns="player_name",
                           values="points_vs_field", aggfunc="mean").round(1)
by_plan["best"] = by_plan.idxmax(axis=1)
by_plan["gap to 2nd"] = (by_plan[list(CANDIDATES)].max(axis=1)
                         - by_plan[list(CANDIDATES)].apply(lambda r: r.nlargest(2).iloc[-1], axis=1)).round(1)
by_plan.sort_values("Jahmyr Gibbs", ascending=False)

player_name,Bijan Robinson,Jahmyr Gibbs,Josh Allen,best,gap to 2nd
composition,,,,,
2QB2RB1TE,116.9,127.0,113.2,Jahmyr Gibbs,10.1
2QB2RB1WR,112.9,122.9,108.8,Jahmyr Gibbs,10.0
2QB1RB1WR1TE,101.7,111.5,97.8,Jahmyr Gibbs,9.8
2QB3RB,100.9,110.7,97.0,Jahmyr Gibbs,9.8
1QB2RB1WR1TE,93.8,104.0,81.1,Jahmyr Gibbs,10.2
1QB3RB1TE,88.7,98.8,75.8,Jahmyr Gibbs,10.1
1QB3RB1WR,84.6,95.2,71.9,Jahmyr Gibbs,10.6
2QB1RB2TE,81.8,91.9,78.0,Jahmyr Gibbs,10.1
2QB1RB2WR,80.8,90.6,76.8,Jahmyr Gibbs,9.8


**The ordering is the same under every opening tested.** There is no plan on which Allen is the
right first pick — not even the two-quarterback openings that most want him, because those are
exactly the plans that can pick up a second quarterback at 28 and 29 instead.

Note also that the *plan* moves the result by far more than the *pick* does: the spread down the
Gibbs column dwarfs the spread across any row. That is the earlier finding restated inside this one.

## What the simulation can and cannot separate

The Gibbs-over-Bijan row had a suspiciously small standard error. Here is why, and it materially
changes how much that 93% is worth.

Both backs are run through the same rooms, on the same plan, at the same pick. Their rosters are
therefore *identical apart from which back is on them*. So the simulated gap between them cannot be
anything other than their projection gap — plus a small correction because whichever back you leave
on the board is one the other thirteen teams get to have.

In [9]:
projected = board.set_index("player_name").projected
projection_gap = projected["Jahmyr Gibbs"] - projected["Bijan Robinson"]

# points_vs_field = mine - field_mean, and the back you pass on lands in the field.
predicted = projection_gap * TEAMS / (TEAMS - 1)
observed = (wide["Jahmyr Gibbs"] - wide["Bijan Robinson"]).mean()

print(f"consensus projection gap, Gibbs - Bijan : {projection_gap:6.2f}")
print(f"  + field correction, x {TEAMS}/{TEAMS - 1}            : {predicted:6.2f}   <- predicted")
print(f"simulated gap, {TRIALS} paired rooms       : {observed:6.2f}   <- observed")
print(f"\nunexplained residual                    : {observed - predicted:6.2f}")

consensus projection gap, Gibbs - Bijan :   9.40
  + field correction, x 14/13            :  10.12   <- predicted
simulated gap, 3000 paired rooms       :  10.15   <- observed

unexplained residual                    :   0.02


**The simulation is not evidence about Bijan versus Gibbs.** It reproduces the projection gap to
within a rounding error, because for two players at the same position on the same plan there is
nothing else for it to measure. The 93% win rate is not 93% confidence that Gibbs is the better
player — it is the fraction of rooms in which nothing else happened to intervene, and it would read
the same if the projection were wrong by fifty points.

This is exactly the comparison the simulation *does* add something to for Allen: he changes the
shape of the roster, so his rows genuinely interact with replacement level, the superflex slot and
the draft's supply curve. Keep the simulation's verdict on the quarterback; go behind it for the
backs.

## So is the projection right? Touchdowns.

The whole Gibbs-over-Bijan case now rests on the consensus projection. That projection is an
extrapolation from 2025, so look at what happened in 2025.

In [10]:
q('''
    SELECT player_display_name AS player_name, count(*) AS games,
           round(sum(rushing_tds) + sum(receiving_tds), 0) AS tds,
           round(sum(rushing_yards) + sum(receiving_yards), 0) AS yards,
           round(sum(receptions), 0) AS receptions,
           round(sum(carries), 0) AS carries,
           round(sum(carries) + sum(receptions), 0) AS touches,
           round((sum(rushing_tds) + sum(receiving_tds)) / (sum(carries) + sum(receptions)), 4)
               AS td_per_touch,
           round(avg(target_share), 3) AS target_share
    FROM weekly_stats
    WHERE season = 2025 AND season_type = 'REG'
      AND player_display_name IN ('Bijan Robinson', 'Jahmyr Gibbs')
    GROUP BY 1 ORDER BY 1
''')

,player_name,games,tds,yards,receptions,carries,touches,td_per_touch,target_share
0,Bijan Robinson,17,11.0,2298.0,79.0,287.0,366.0,0.0301,0.198
1,Jahmyr Gibbs,17,18.0,1839.0,77.0,243.0,320.0,0.0563,0.162


Gibbs outscored Bijan on **fewer yards, fewer carries, fewer touches and a smaller target share.**
The entire difference, and more, is touchdowns: eighteen against eleven.

Touchdowns are the least repeatable thing a running back does. How much of a touchdown-rate edge
actually survives into the next season, across every back since 2010 with a real workload:

In [11]:
pairs = q('''
    WITH seasons AS (
        SELECT player_id, player_display_name AS player_name, season,
               sum(carries) + sum(receptions) AS touches,
               sum(rushing_tds) + sum(receiving_tds) AS tds,
               sum(rushing_yards) + sum(receiving_yards) AS yards
        FROM weekly_stats
        WHERE season_type = 'REG' AND position = 'RB' AND season BETWEEN 2010 AND 2025
        GROUP BY 1, 2, 3
        HAVING sum(carries) + sum(receptions) >= 150
    )
    SELECT this.player_name, this.season,
           this.tds / this.touches AS rate, next.tds / next.touches AS rate_next,
           this.yards / this.touches AS ypt, next.yards / next.touches AS ypt_next,
           this.touches
    FROM seasons this JOIN seasons next
      ON this.player_id = next.player_id AND next.season = this.season + 1
''')

rows = []
for floor in (150, 200, 250):
    cohort = pairs[pairs.touches >= floor]
    r, p = stats.pearsonr(cohort.rate, cohort.rate_next)
    slope, _ = np.polyfit(cohort.rate, cohort.rate_next, 1)
    rows.append({"cohort": f"touches >= {floor}", "n": len(cohort), "r": round(r, 3),
                 "p": f"{p:.3f}", "share of edge that persists": f"{slope:.0%}"})
persistence = pd.DataFrame(rows)

r_yards, _ = stats.pearsonr(pairs.ypt, pairs.ypt_next)
print(f"league mean TD per touch          : {pairs.rate.mean():.4f}")
print(f"yards per touch, year over year   : r = {r_yards:.3f}   <- for contrast")
persistence

league mean TD per touch          : 0.0343
yards per touch, year over year   : r = 0.384   <- for contrast


,cohort,n,r,p,share of edge that persists
0,touches >= 150,235,0.197,0.002,21%
1,touches >= 200,186,0.211,0.004,23%
2,touches >= 250,116,0.255,0.006,29%


**Touchdown rate barely persists** — about a fifth of an edge carries into the next year, and it
predicts about half as well as yards per touch, the thing Bijan was better at. So apply that
regression to what each back did in 2025:

In [12]:
slope, intercept = np.polyfit(pairs.rate, pairs.rate_next, 1)
league_rate = pairs.rate.mean()

actual = q('''
    SELECT player_display_name AS player_name,
           sum(rushing_tds) + sum(receiving_tds) AS tds,
           sum(carries) + sum(receptions) AS touches
    FROM weekly_stats
    WHERE season = 2025 AND season_type = 'REG'
      AND player_display_name IN ('Bijan Robinson', 'Jahmyr Gibbs')
    GROUP BY 1
''').set_index("player_name")

TD_POINTS = float(settings.rush_td_pts)
scored = q('''
    SELECT player_name, round(league_points, 1) AS points_2025
    FROM points_over_replacement
    WHERE league_key = ? AND season = 2025
      AND player_name IN ('Bijan Robinson', 'Jahmyr Gibbs')
''', [LEAGUE]).set_index("player_name")

regression = pd.DataFrame({
    "tds_2025": actual.tds,
    "td_per_touch": (actual.tds / actual.touches).round(4),
    "expected_2026_rate": (intercept + slope * actual.tds / actual.touches).round(4),
    "tds_at_that_rate": ((intercept + slope * actual.tds / actual.touches) * actual.touches).round(1),
    "td_neutral_2025_tds": (league_rate * actual.touches).round(1),
})
regression["points_2025"] = scored.points_2025
regression["td_neutral_2025_points"] = (
    regression.points_2025 - (regression.tds_2025 - regression.td_neutral_2025_tds) * TD_POINTS
).round(1)

# Which way is the consensus moving them, relative to what they actually did?
actual_2025 = scored.points_2025["Bijan Robinson"] - scored.points_2025["Jahmyr Gibbs"]
projected_2026 = projected["Bijan Robinson"] - projected["Jahmyr Gibbs"]
print(f"Bijan - Gibbs, 2025 actual points   : {actual_2025:+6.1f}  (Bijan ahead)")
print(f"Bijan - Gibbs, 2026 consensus       : {projected_2026:+6.1f}")
print(f"consensus swing toward Gibbs        : {actual_2025 - projected_2026:6.1f} pts")
print()
regression

Bijan - Gibbs, 2025 actual points   :   +2.9  (Bijan ahead)
Bijan - Gibbs, 2026 consensus       :   -9.4
consensus swing toward Gibbs        :   12.3 pts



,tds_2025,td_per_touch,expected_2026_rate,tds_at_that_rate,td_neutral_2025_tds,points_2025,td_neutral_2025_points
player_name,,,,,,,
Jahmyr Gibbs,18.0,0.0562,0.0394,12.6,11.0,328.4,286.4
Bijan Robinson,11.0,0.0301,0.0340,12.4,12.5,331.3,340.3


Two readings of the same fact, and they agree.

Projected forward, the two backs are expected to score **the same number of touchdowns** in 2026 —
Gibbs falls, Bijan rises, and they meet. Looked at backwards, a touchdown-neutral 2025 turns Bijan's
narrow win into a wide one.

And that is the part the consensus cannot be doing: Bijan **outscored** Gibbs in 2025, yet is
projected **below** him in 2026. The market is moving points toward Gibbs, in the season after the
touchdown rate that earned them.

The warehouse's own preferred feature says the same thing without any of this arithmetic. Its
multi-year baselines carry a touchdown-regressed points-per-game alongside the raw one:

In [13]:
q('''
    SELECT player_name, seasons_used, games_used,
           round(weighted_ppg_ppr, 2) AS raw_ppg,
           round(weighted_td_regressed_ppg, 2) AS td_regressed_ppg,
           round(weighted_snap_share, 3) AS snap_share,
           round(weighted_target_share, 3) AS target_share,
           round(weighted_carries_per_game, 1) AS carries_pg,
           round(weighted_receptions_per_game, 2) AS receptions_pg,
           round(weighted_games_per_season, 1) AS games_per_season
    FROM player_weighted_baselines
    WHERE target_season = 2026 AND player_name IN ('Bijan Robinson', 'Jahmyr Gibbs')
    ORDER BY weighted_td_regressed_ppg DESC
''')

,player_name,seasons_used,games_used,raw_ppg,td_regressed_ppg,snap_share,target_share,carries_pg,receptions_pg,games_per_season
0,Bijan Robinson,3,51.0,19.07,19.55,0.742,0.171,15.9,3.93,17.0
1,Jahmyr Gibbs,3,49.0,19.89,18.00,0.599,0.138,13.8,3.72,16.4


**Gibbs leads on raw points per game and trails on touchdown-regressed points per game.** That
reversal is the whole argument in one row — and touchdown-regressed prior scoring is the single best
feature this warehouse has found for predicting next-season production.

Bijan also plays more: more snaps, more carries, more targets, more games. Gibbs shares a backfield;
Bijan does not.

## Everything that isn't the projection favours Bijan

Collecting the tiebreakers, none of which the consensus projection is weighing heavily.

In [14]:
tiebreak = board.set_index("player_name").loc[["Bijan Robinson", "Jahmyr Gibbs"]]
baselines = q('''
    SELECT player_name, round(weighted_td_regressed_ppg, 2) AS td_regressed_ppg,
           round(weighted_snap_share, 3) AS snap_share
    FROM player_weighted_baselines
    WHERE target_season = 2026 AND player_name IN ('Bijan Robinson', 'Jahmyr Gibbs')
''', ).set_index("player_name")

comparison = pd.DataFrame({
    "consensus projection": tiebreak.projected,
    "in-house model": tiebreak.inhouse_points.round(1),
    "td-regressed ppg": baselines.td_regressed_ppg,
    "snap share": baselines.snap_share,
    "offensive line": tiebreak.ol_grade,
    "ol tier": tiebreak.ol_tier,
    "durability": tiebreak.availability,
    "bye week": tiebreak.bye_week,
    "adp": tiebreak.consensus_adp,
}).T
comparison

player_name,Bijan Robinson,Jahmyr Gibbs
consensus projection,305.3,314.7
in-house model,286.2,284.7
td-regressed ppg,19.55,18.0
snap share,0.742,0.599
offensive line,83.9,66.1
ol tier,Q4 best,Q3
durability,0.97,0.95
bye week,11,6
adp,2.6,1.8


The consensus projection is the **only** line on which Gibbs leads. The in-house model, the
touchdown-regressed baseline, snap share, the offensive line in front of him and the durability
estimate all point the other way — and Bijan is the marginally *cheaper* pick by ADP, so taking him
is not even paying a premium.

Both are 23, both "Proven Prime", both elite finishers in each of the last three seasons: the
archetype work does not separate them either.

## What would have to move to change the answer

Two break-evens, both computable from the decomposition above: the simulated gap between two
same-position candidates is just their projection gap scaled by teams/(teams - 1).

In [15]:
gaps = {
    "Bijan catches Gibbs": projection_gap,
    "Allen catches Gibbs": (wide["Jahmyr Gibbs"] - wide["Josh Allen"]).mean() * (TEAMS - 1) / TEAMS,
}
td_swing = (regression.loc["Jahmyr Gibbs", "tds_2025"]
            - regression.loc["Jahmyr Gibbs", "tds_at_that_rate"]) * TD_POINTS
bijan_swing = (regression.loc["Bijan Robinson", "tds_at_that_rate"]
               - regression.loc["Bijan Robinson", "tds_2025"]) * TD_POINTS

print(f"projected points Bijan must gain on Gibbs to tie : {gaps['Bijan catches Gibbs']:6.1f}")
print(f"projected points Allen must gain to tie          : {gaps['Allen catches Gibbs']:6.1f}")
print()
print(f"points of Gibbs' 2025 the TD regression removes  : {td_swing:6.1f}")
print(f"points of Bijan's 2025 the TD regression adds    : {bijan_swing:6.1f}")
print(f"combined swing toward Bijan                      : {td_swing + bijan_swing:6.1f}")

projected points Bijan must gain on Gibbs to tie :    9.4
projected points Allen must gain to tie          :   12.7

points of Gibbs' 2025 the TD regression removes  :   32.4
points of Bijan's 2025 the TD regression adds    :    8.4
combined swing toward Bijan                      :   40.8


**Bijan needs to close single digits; touchdown regression alone is worth several times that.**
That swing is measured against 2025 actuals rather than against the 2026 projections, so some of it
may already be priced in — but the swing the consensus actually applied runs the *other* way, toward
Gibbs. A gap this small, resting on the least repeatable thing a back does, is not enough to
separate them: the honest reading is "level", not "Gibbs".

Allen's gap is the larger of the two and there is no comparable known bias pushing back on it. Two
effects the model omits do lean his way — waiver running backs appear all season and waiver
quarterbacks do not, and the durability estimate is built from injured-reserve stints so it misses
one- and two-game absences — but neither is worth the margin.

## The verdict

In [16]:
verdict = pd.DataFrame([
    {"rank": 1, "player": "Bijan Robinson",
     "case": "level with Gibbs on projection once TDs regress; ahead on everything else"},
    {"rank": 1, "player": "Jahmyr Gibbs",
     "case": "leads the consensus board, but the lead is a touchdown rate that does not persist"},
    {"rank": 3, "player": "Josh Allen",
     "case": "best player, wrong pick: superflex sells a starting QB at 28, not a back like these"},
]).set_index("rank")

print("Take a running back. Bijan and Gibbs are a coin flip — take Bijan if you want the")
print("workload and the offensive line, Gibbs if you trust the consensus board.\n")
verdict

Take a running back. Bijan and Gibbs are a coin flip — take Bijan if you want the
workload and the offensive line, Gibbs if you trust the consensus board.



,player,case
rank,,
1,Bijan Robinson,level with Gibbs on projection once TDs regress; ahead on everything else
1,Jahmyr Gibbs,"leads the consensus board, but the lead is a touchdown rate that does not persist"
3,Josh Allen,"best player, wrong pick: superflex sells a starting QB at 28, not a back like these"


## What this does not model

- **In-season replacement is not symmetric.** Running backs reach waivers all year; startable
  quarterbacks in a superflex league do not. This is the strongest omitted argument for Allen.
- **Durability comes from injured-reserve stints**, so it counts season-ending injuries and misses
  the one- and two-game absences that decide close weeks.
- **Availability probabilities are marginal**, one player at a time, and the swap table treats them
  as independent. The simulation does not — it drafts a real board — which is why the two disagree
  on the size of the quarterback's deficit while agreeing on its sign.
- **The ADP board is a superflex board but the projections are not re-priced by format**, the known
  gap flagged in `draft_strategy.ipynb`.
- **Keepers, trades and in-season churn are absent.** This is a season-total, draft-day model.

One housekeeping note. `draft_board.ipynb` carries an older version of this comparison whose
committed outputs predate the half-PPR reception fix; re-running it against the current warehouse
reverses its Bijan-over-Gibbs ordering and moves its plan table. Run `scripts/run_notebooks.sh`
before trusting its numbers against these.